# Trading Intelligent — CH01
## Market Data, Returns and First Diagnostics

**Auteur académique : Tawfik Masrour**

Ce notebook accompagne le chapitre **Comprendre le marché et ses données**. Il construit les objets de base utilisés ensuite : prix, OHLCV, bid/ask, rendements, volatilité glissante et diagnostics de qualité.

### 1. Configuration et reproductibilité

Le jeu pédagogique porte l'identifiant **TI_CH01_MARKET_V1** et le nom `TI_CH01_Market_Data_v1.csv`. Il est synthétique et déterministe : si le fichier n'est pas disponible, le notebook reconstruit exactement la même série.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATASET_ID = "TI_CH01_MARKET_V1"
DATASET_FILENAME = "TI_CH01_Market_Data_v1.csv"
SEED = 20260929
np.random.seed(SEED)
pd.set_option("display.max_columns", 20)

def generate_ti_ch01_market_v1(seed=SEED):
    rng = np.random.default_rng(seed)
    dates = pd.bdate_range("2023-01-02", periods=756)
    n = len(dates)
    vol = np.full(n, 0.010)
    vol[170:260] = 0.022
    vol[260:430] = 0.009
    vol[430:515] = 0.028
    vol[515:] = 0.012
    vol = pd.Series(vol).rolling(7, center=True, min_periods=1).mean().to_numpy()
    mu = np.full(n, 0.00020)
    mu[430:515] = -0.00035
    z = rng.standard_t(df=6, size=n) / np.sqrt(6/(6-2))
    log_ret = mu + vol*z
    close = 100*np.exp(np.cumsum(log_ret))
    overnight = rng.normal(0, vol*0.22)
    open_ = np.empty(n)
    open_[0] = 100*np.exp(overnight[0])
    open_[1:] = close[:-1]*np.exp(overnight[1:])
    range_scale = np.maximum(vol, np.abs(log_ret))*(0.8+0.6*rng.random(n))
    high = np.maximum(open_, close)*(1+range_scale*(0.45+0.35*rng.random(n)))
    low = np.minimum(open_, close)*(1-range_scale*(0.45+0.35*rng.random(n)))
    low = np.maximum(low, 0.01)
    volume = np.exp(16.3 + 11*np.abs(log_ret) + rng.normal(0,0.28,n)).astype("int64")
    spread_bps = 3.0 + 85*vol + 35*np.abs(log_ret) + rng.gamma(1.5,0.4,n)
    mid = close.copy()
    spread = mid*spread_bps/10_000
    return pd.DataFrame({"date":dates,"symbol":"SYNTH-MKT","open":open_,"high":high,"low":low,"close":close,"adjusted_close":close,"volume":volume,"bid":mid-spread/2,"ask":mid+spread/2})

def locate_dataset(filename=DATASET_FILENAME):
    cwd=Path.cwd(); roots=[cwd,cwd.parent,cwd.parent.parent]; subdirs=[Path('.'),Path('data'),Path('datasets'),Path('resources')]
    for root in roots:
        for subdir in subdirs:
            candidate=(root/subdir/filename).resolve()
            if candidate.exists(): return candidate
    return None

DATA_PATH=locate_dataset()
if DATA_PATH is not None:
    df=pd.read_csv(DATA_PATH,parse_dates=['date']); print(f"{DATASET_ID} chargé depuis : {DATA_PATH}")
else:
    df=generate_ti_ch01_market_v1(); print(f"{DATASET_ID} absent localement : version déterministe régénérée en mémoire.")
print('Dimensions :',df.shape)


### 2. Contrôler le schéma

**OHLCV** signifie *Open, High, Low, Close, Volume*. Une ligne résume une période par son prix d'ouverture, son plus haut, son plus bas, son prix de clôture et son volume.

In [ ]:
df=df.sort_values('date').reset_index(drop=True)
display(df.head())
assert df['date'].is_monotonic_increasing
assert not df['date'].duplicated().any()
assert (df['high']>=df[['open','close']].max(axis=1)).all()
assert (df['low']<=df[['open','close']].min(axis=1)).all()
assert (df['ask']>=df['bid']).all()
print('Contrôles de cohérence : OK')


### 3. Prix, bid, ask et spread

Le **bid** est le meilleur prix acheteur, l'**ask** le meilleur prix vendeur et le **spread** leur écart. On l'exprime souvent en **points de base** (*basis points*, bps), où 1 bps = 0,01 %.

In [ ]:
mid=(df['bid']+df['ask'])/2
spread=df['ask']-df['bid']
df['spread_bps']=10_000*spread/mid
display(df[['date','bid','ask','spread_bps']].head())
print('Spread médian (bps) :',round(df['spread_bps'].median(),2))


### 4. Du prix au rendement

Le rendement simple est $R_t=P_t/P_{t-1}-1$ et le rendement logarithmique est $r_t=\log(P_t/P_{t-1})$. Pour de petites variations, ils sont proches, mais leurs règles de composition diffèrent.

In [ ]:
df['return_simple']=df['close'].pct_change()
df['return_log']=np.log(df['close']).diff()
display(df[['date','close','return_simple','return_log']].head(8))


### 5. Courbe de richesse et volatilité

Une unité monétaire investie évolue par composition des rendements. Une première estimation de volatilité utilise l'écart-type des rendements sur une fenêtre glissante.

In [ ]:
df['wealth']=(1+df['return_simple'].fillna(0)).cumprod()
df['vol20_ann']=df['return_log'].rolling(20).std()*np.sqrt(252)
fig,ax=plt.subplots(figsize=(10,4)); ax.plot(df['date'],df['wealth']); ax.set(title="Valeur d'une unité monétaire investie",xlabel='Date',ylabel='Richesse relative'); ax.grid(alpha=.2); plt.show()
fig,ax=plt.subplots(figsize=(10,4)); ax.plot(df['date'],100*df['vol20_ann']); ax.set(title='Volatilité glissante annualisée (20 jours)',xlabel='Date',ylabel='Volatilité (%)'); ax.grid(alpha=.2); plt.show()


### 6. Premiers faits stylisés

Les rendements fluctuent autour d'un niveau plus stable que les prix, tandis que les fortes amplitudes tendent à apparaître par grappes (*volatility clustering*).

In [ ]:
fig,ax=plt.subplots(figsize=(10,4)); ax.plot(df['date'],100*df['return_log'],lw=.7); ax.axhline(0,lw=.8); ax.set(title='Rendements logarithmiques',xlabel='Date',ylabel='Rendement (%)'); ax.grid(alpha=.2); plt.show()


### 7. Mini-défi

1. Calculez un rendement simple à 5 jours sans utiliser `pct_change(5)`, puis vérifiez votre formule.
2. Comparez le spread moyen durant les 10 % de journées les plus volatiles au reste de l'échantillon.
3. Construisez une colonne `direction` valant 1 si le rendement du lendemain est positif et 0 sinon. Pourquoi cette colonne ne doit-elle jamais être utilisée comme feature au temps $t$ ?
4. Expliquez pourquoi la moyenne du prix n'a pas la même interprétation que la moyenne du rendement.

In [ ]:
# Votre travail commence ici.
# Q1

# Q2

# Q3

# Q4 : ajoutez une cellule Markdown pour votre réponse.


### 8. À retenir

- Le temps est constitutif du problème.
- Open, High, Low, Close, Volume, Bid et Ask représentent des objets différents.
- Un prix et un rendement ne répondent pas à la même question.
- Le spread matérialise déjà un coût implicite.
- Avant toute IA : vérifier provenance, calendrier, fréquence, unités et qualité des données.

**Suite :** stationnarité, bruit blanc, autocorrélation, AR et MA.